# Training the agents, forecast-window ablation and state/spill-penalty factorial

In [ ]:
from pathlib import Path
import os, json, gzip, re, pickle, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict, deque
from scipy.optimize import linprog, milp, LinearConstraint, Bounds

try:
    import gymnasium as gym
    from gymnasium import spaces
except Exception:
    import gym
    from gym import spaces

import torch
import torch.nn as nn
import torch.nn.functional as F

plt.style.use('seaborn-v0_8-whitegrid')
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

ED_LEVELS = [round(x * 0.05, 2) for x in range(1, 21)]
print(f"ED levels: {[f'{int(l*100)}%' for l in ED_LEVELS]}")

In [ ]:
SMOKE = os.environ.get("UC_SMOKE") == "1"
PPO_SCALE = 0.01 if SMOKE else 1.0
N_EVAL = 3 if SMOKE else None
OUT = Path("outputs")
for sub_dir in ("saved_models_ppo", "saved_models_dqn", "ablation"):
    (OUT / sub_dir).mkdir(parents=True, exist_ok=True)
print("quick-check mode" if SMOKE else "full run", "| outputs ->", OUT.resolve())

## 1. Data loading

In [ ]:
def _first_existing(folder, names, required=True):
    folder = Path(folder)
    for n in names:
        if (folder / n).exists(): return folder / n
    if required: raise FileNotFoundError(f"None of {names} in {folder}")
    return None

def _read_json(path):
    path = Path(path)
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f: return json.load(f)

def _norm_col(df, name):
    df = df.copy()
    if len(df.columns) > 0 and str(df.columns[0]).lower().startswith("unnamed"):
        df = df.rename(columns={df.columns[0]: name})
    return df

def _load_profiles_and_ic(folder):
    folder = Path(folder)
    if (folder / "profiles.csv").exists():
        return (_norm_col(pd.read_csv(folder/"profiles.csv"),"hour"),
                _norm_col(pd.read_csv(folder/"initial_conditions.csv"),"generator"))
    xls = pd.ExcelFile(folder/"explanatory_variables.xlsx")
    def find(kws):
        for k in kws:
            for s in xls.sheet_names:
                if all(x.lower() in s.lower() for x in k): return s
        return xls.sheet_names[0]
    return (_norm_col(pd.read_excel(folder/"explanatory_variables.xlsx",
                                    sheet_name=find([("profile",)])),"hour"),
            _norm_col(pd.read_excel(folder/"explanatory_variables.xlsx",
                                    sheet_name=find([("initial",)])),"generator"))

def get_profile_series(df):
    def fc(kws):
        for c in df.columns:
            if all(k in str(c).lower() for k in kws): return c
        return None
    d = pd.to_numeric(df[fc(["demand"])],errors="coerce").fillna(0).to_numpy(float)
    w = pd.to_numeric(df[fc(["wind"])],errors="coerce").fillna(0).to_numpy(float) if fc(["wind"]) else np.zeros_like(d)
    s = pd.to_numeric(df[fc(["solar"])],errors="coerce").fillna(0).to_numpy(float) if fc(["solar"]) else np.zeros_like(d)
    return d, w, s, d-w-s

def list_instance_dirs(root):
    dirs = sorted(p for p in Path(root).rglob("instance_*") if p.is_dir())
    if not dirs: raise FileNotFoundError(f"No instance_* under {root}")
    return dirs

def split_instance_dirs(dirs, val_quarters=None):
    val_quarters = val_quarters or set()
    train, valid = [], []
    for d in dirs:
        m = re.match(r"instance_(\d{4})_(Q[1-4])_(\d+)", d.name)
        if m and (int(m.group(1)), m.group(2)) in val_quarters: valid.append(d)
        else: train.append(d)
    return train, valid

def load_single_bundle(d):
    d = Path(d)
    profiles, ic = _load_profiles_and_ic(d)
    inp = _read_json(_first_existing(d,["InputData.json","InputData.json.gz"]))
    outp_path = _first_existing(d,["OutputData.json","OutputData.json.gz"],required=False)
    outp = _read_json(outp_path) if outp_path else None
    return {"instance_id":d.name,"profiles":profiles,"initial_conditions":ic,
            "input_json":inp,"output_json":outp}

def _as_float_array(v):
    if v is None: return np.array([], float)
    return pd.to_numeric(pd.Series(v if isinstance(v,list) else [v]),
                          errors="coerce").dropna().to_numpy(float)

def approx_marginal_cost(mw, cost):
    mw, cost = _as_float_array(mw), _as_float_array(cost)
    if len(mw) < 2: return 0.0
    return float((cost[1]-cost[0])/max(mw[1]-mw[0],1e-9))

def parse_generators(inp):
    gen = pd.DataFrame.from_dict(inp["Generators"],orient="index").reset_index().rename(columns={"index":"generator"})
    gen["Type_clean"] = gen["Type"].str.strip().str.lower()
    th = gen[gen["Type_clean"]=="thermal"].copy()
    if th.empty: raise ValueError("No thermal generators")
    th["cost_curve_mw"] = th["Production cost curve (MW)"].apply(_as_float_array)
    th["cost_curve_usd"] = th["Production cost curve ($)"].apply(_as_float_array)
    th["pmin"] = th["cost_curve_mw"].apply(lambda x: float(x[0]) if len(x) else 0.0)
    th["pmax"] = th["cost_curve_mw"].apply(lambda x: float(x[-1]) if len(x) else 0.0)
    th["marginal_cost"] = th.apply(lambda r: approx_marginal_cost(r["cost_curve_mw"],r["cost_curve_usd"]),axis=1)
    th["startup_cost"] = th["Startup costs ($)"].apply(
        lambda x: float(_as_float_array(x)[0]) if len(_as_float_array(x)) else 0.0)
    for src,tgt in [("Initial power (MW)","init_power"),("Initial status (h)","init_status"),
                    ("Minimum uptime (h)","min_up"),("Minimum downtime (h)","min_down"),
                    ("Ramp up limit (MW)","ramp_up"),("Ramp down limit (MW)","ramp_down")]:
        th[tgt] = pd.to_numeric(th.get(src,0),errors="coerce").fillna(0.0)
    th = th.sort_values(["marginal_cost","startup_cost","pmax"]).reset_index(drop=True)
    return th[["generator","cost_curve_mw","cost_curve_usd","pmin","pmax",
               "marginal_cost","startup_cost","init_power","init_status",
               "min_up","min_down","ramp_up","ramp_down"]]

def get_balance_penalty(inp, default=12533.0):
    params = inp.get("Parameters",{})
    for k in ["Power balance penalty ($/MW)","Power balance penalty"]:
        if k in params:
            v = pd.to_numeric(pd.Series([params[k]]),errors="coerce").iloc[0]
            if pd.notna(v): return float(v)
    return default

def bundle_to_episode(b):
    th = parse_generators(b["input_json"])
    d, w, s, nd = get_profile_series(b["profiles"])
    return {"instance_id":b["instance_id"],"thermal_df":th,
            "demand":d,"wind":w,"solar":s,"net_demand":nd,
            "output_json":b["output_json"],
            "balance_penalty":get_balance_penalty(b["input_json"])}

def validate_compat(eps):
    ref = eps[0]["thermal_df"]["generator"].tolist()
    for e in eps[1:]:
        if e["thermal_df"]["generator"].tolist() != ref:
            raise ValueError(f"Mismatch: {e['instance_id']}")
    return True

def load_episodes(dirs, max_n=None, seed=42, verbose=True):
    dirs = list(dirs)
    np.random.default_rng(seed).shuffle(dirs)
    if max_n: dirs = dirs[:max_n]
    eps, fails = [], []
    for d in dirs:
        try: eps.append(bundle_to_episode(load_single_bundle(d)))
        except Exception as e: fails.append((str(d),repr(e)))
    if not eps: raise RuntimeError(f"No episodes. Failures: {fails[:3]}")
    validate_compat(eps)
    if verbose:
        print(f"Loaded {len(eps)} | Failed {len(fails)} | "
              f"Generators: {len(eps[0]['thermal_df'])} | Horizon: {len(eps[0]['demand'])}h")
    return eps

In [ ]:
DATA_ROOT  = Path(__import__("os").environ.get("UC_DATA_ROOT", "../rl_ready_competition_data"))
TRAIN_ROOT = DATA_ROOT / "train"
N_TRAIN, N_VALID, N_TEST = 300, 25, 100

all_dirs = list_instance_dirs(TRAIN_ROOT)
train_dirs, valid_dirs = split_instance_dirs(all_dirs, {(2026,"Q4"),(2027,"Q4"),(2028,"Q4"),(2029,"Q4")})
train_episodes = load_episodes(train_dirs, max_n=N_TRAIN, seed=42)
valid_episodes = load_episodes(valid_dirs, max_n=N_VALID, seed=43)

used_ids  = {e["instance_id"] for e in train_episodes} | {e["instance_id"] for e in valid_episodes}
test_dirs = [d for d in all_dirs if d.name not in used_ids]
test_episodes = load_episodes(test_dirs, max_n=N_TEST, seed=99)

## 2. Environment

In [ ]:
class UCOnlyEnv:

    def __init__(self, episodes, spill_penalty=8000.0, reward_scale=1e5,
                 random_reset=True, seed=42, lookahead=8,
                 state_version="v2", obs_window=72):
        validate_compat(episodes)
        self.episodes = list(episodes)
        self.spill_penalty = spill_penalty
        self.reward_scale = reward_scale
        self.random_reset = random_reset
        self.rng = np.random.default_rng(seed)
        self.lookahead = lookahead
        self.state_version = state_version
        self.obs_window = int(obs_window)

        th = episodes[0]["thermal_df"].reset_index(drop=True)
        self.generator_names = th["generator"].tolist()
        self.n_gen = len(th)
        self.horizon = len(episodes[0]["demand"])
        self.pmin = th["pmin"].to_numpy(float)
        self.pmax = th["pmax"].to_numpy(float)
        self.marginal_cost = th["marginal_cost"].to_numpy(float)
        self.startup_cost_arr = th["startup_cost"].to_numpy(float)
        self.min_up = th["min_up"].to_numpy(float)
        self.min_down = th["min_down"].to_numpy(float)
        self.ramp_up = th["ramp_up"].to_numpy(float)
        self.ramp_down = th["ramp_down"].to_numpy(float)
        self.cost_curve_mw = th["cost_curve_mw"].tolist()
        self.cost_curve_usd = th["cost_curve_usd"].tolist()
        self.thermal_cap = float(self.pmax.sum())

        self._mu_scale = max(float(self.min_up.max()), 1.0)
        self._md_scale = max(float(self.min_down.max()), 1.0)

        if self.state_version == "v1":
            self.obs_dim = 1 + self.n_gen + self.horizon + self.n_gen + 4
        else:
            self.obs_dim = 3 + self.obs_window + 4 + 3 * self.n_gen

        self.reset(options={"episode_index": 0})

    @property
    def obs_layout(self):
        n, W = self.n_gen, self.obs_window
        if self.state_version == "v1":
            H = self.horizon
            return {"time": (0, 1), "prev_on": (1, 1 + n),
                    "net_demand": (1 + n, 1 + n + H),
                    "status": (1 + n + H, 1 + n + H + n),
                    "lookahead": (1 + n + H + n, 1 + n + H + n + 4)}
        o = 0
        lay = {}
        lay["time"] = (o, o + 3); o += 3
        lay["window"] = (o, o + W); o += W
        lay["lookahead"] = (o, o + 4); o += 4
        lay["on_off"] = (o, o + n); o += n
        lay["can_start"] = (o, o + n); o += n
        lay["can_stop"] = (o, o + n); o += n
        return lay

    def reset(self, seed=None, options=None):
        if seed: self.rng = np.random.default_rng(seed)
        options = options or {}
        idx = int(options.get("episode_index",
                  self.rng.integers(len(self.episodes)) if self.random_reset else 0))
        self.ep_idx = idx
        ep = self.episodes[idx]
        self.demand = np.asarray(ep["demand"], float)
        self.wind   = np.asarray(ep["wind"], float)
        self.solar  = np.asarray(ep["solar"], float)
        self.net_demand = self.demand - self.wind - self.solar
        self.balance_penalty = float(ep.get("balance_penalty", 12533.0))

        th = ep["thermal_df"]
        self.current_dispatch = th["init_power"].to_numpy(float).copy()
        self.status_duration  = th["init_status"].to_numpy(float).copy()
        self.prev_on = (self.current_dispatch > 1e-6).astype(int)
        self.t = 0
        return self._obs(), {"instance_id": ep["instance_id"]}

    def _constraint_clocks(self):
        sd = self.status_duration
        is_on = (self.current_dispatch > 1e-6) | (sd > 0)

        hours_off = np.where(sd < 0, -sd, 0.0)
        can_start = np.where(is_on, 0.0, np.maximum(0.0, self.min_down - hours_off))

        hours_on = np.where(sd > 0, sd, 0.0)
        can_stop = np.where(~is_on, 0.0, np.maximum(0.0, self.min_up - hours_on))

        return (can_start / self._md_scale,
                can_stop / self._mu_scale,
                is_on.astype(float))

    def _demand_window(self):
        cap = max(self.thermal_cap, 1.0)
        W = self.obs_window
        seg = self.net_demand[self.t:self.t + W] / cap
        if len(seg) < W:
            pad = np.full(W - len(seg), seg[-1] if len(seg) else 0.0)
            seg = np.concatenate([seg, pad])
        return seg.astype(np.float32)

    def _lookahead_stats(self):
        cap = max(self.thermal_cap, 1.0)
        L = self.lookahead
        window = self.net_demand[self.t:min(self.t + L, self.horizon)]
        if len(window) < 2:
            return np.zeros(4, dtype=np.float32)
        return np.array([
            float(window.min()) / cap,
            float(window.max()) / cap,
            float(window.mean()) / cap,
            float(window[-1] - window[0]) / (cap * L),
        ], dtype=np.float32)

    def _obs(self):
        cap = max(self.thermal_cap, 1.0)

        if self.state_version == "v1":
            status_norm = np.clip(self.status_duration / 10.0, -1.0, 1.0)
            return np.concatenate([
                [self.t / max(self.horizon - 1, 1)],
                self.prev_on.astype(float),
                self.net_demand / cap,
                status_norm,
                self._lookahead_stats(),
            ]).astype(np.float32)

        h = self.t % 24
        ang = 2.0 * np.pi * h / 24.0
        remaining = (self.horizon - self.t) / max(self.horizon, 1)

        can_start, can_stop, on_off = self._constraint_clocks()
        return np.concatenate([
            [np.sin(ang), np.cos(ang), remaining],
            self._demand_window(),
            self._lookahead_stats(),
            on_off,
            can_start,
            can_stop,
        ]).astype(np.float32)

    def _legalise(self, desired):
        on = np.asarray(desired, int).copy()
        cur_on = (self.current_dispatch > 1e-6) | (self.status_duration > 0)
        on[cur_on & (self.status_duration > 0) & (self.status_duration < self.min_up)] = 1
        on[(~cur_on) & (self.status_duration < 0) & (-self.status_duration < self.min_down)] = 0
        on[cur_on & (on == 0) & (self.current_dispatch > self.ramp_down + 1e-6)] = 1
        return on

    def _update_durations(self, on):
        nxt = np.zeros(self.n_gen, float)
        for i in range(self.n_gen):
            if on[i] == 1:
                nxt[i] = max(1.0, self.status_duration[i] + 1.0) if self.prev_on[i] == 1 else 1.0
            else:
                nxt[i] = min(-1.0, self.status_duration[i] - 1.0) if self.prev_on[i] == 0 else -1.0
        return nxt

    def step(self, action):
        raw = np.asarray(action, int).ravel()[:self.n_gen]
        on  = self._legalise(raw)

        nd_t     = float(self.net_demand[self.t])
        required = max(0.0, nd_t)

        total_pmin_on = float(self.pmin[on == 1].sum())
        total_pmax_on = float(self.pmax[on == 1].sum())

        cur_on = (self.current_dispatch > 1e-6) | (self.status_duration > 0)
        startup_cost = float(np.sum(((on == 1) & (~cur_on)) * self.startup_cost_arr))
        fuel_proxy = float(np.sum(on * self.pmin * self.marginal_cost))
        unserved = max(0.0, required - total_pmax_on)
        over_commit = max(0.0, total_pmin_on - required)

        spill_cost = self.spill_penalty * over_commit

        if required > 0:
            on_indices = np.where(on == 1)[0]
            sorted_by_pmin = on_indices[np.argsort(-self.pmax[on_indices])]
            cumsum = 0.0
            needed = set()
            for idx in sorted_by_pmin:
                if cumsum < required * 1.15:
                    cumsum += self.pmax[idx]
                    needed.add(idx)
            n_excess = len(on_indices) - len(needed)
            excess_unit_penalty = n_excess * 100.0
        else:
            excess_unit_penalty = float(on.sum()) * 500.0

        lookahead_penalty = 0.0
        L = self.lookahead
        t_end = min(self.t + L, self.horizon)
        future_min_demand = float(self.net_demand[self.t:t_end].min())
        if future_min_demand < required * 0.5 and over_commit > 0:
            lookahead_penalty = over_commit * 500.0

        cost = (fuel_proxy + startup_cost
                + self.balance_penalty * unserved
                + spill_cost + excess_unit_penalty + lookahead_penalty)
        reward = -cost / self.reward_scale

        dur_next = self._update_durations(on)
        self.current_dispatch = on.astype(float) * self.pmin

        info = {
            "instance_id": self.episodes[self.ep_idx]["instance_id"],
            "hour": self.t, "net_demand": nd_t,
            "on": on.copy(), "raw_action": raw.copy(),
            "n_overrides": int(np.sum(on != raw)),
            "startup_cost": startup_cost, "fuel_proxy": fuel_proxy,
            "cost": cost, "reward": reward,
            "unserved_proxy": unserved, "over_commit": over_commit,
            "n_startups": int(((on == 1) & (~cur_on)).sum()),
            "n_online": int(on.sum()),
        }
        self.prev_on = on.copy()
        self.status_duration = dur_next
        self.t += 1
        done = self.t >= self.horizon
        obs  = np.zeros(self.obs_dim, np.float32) if done else self._obs()
        return obs, reward, done, False, info

In [ ]:
def _reset(env_obj, idx=None):
    out = env_obj.reset(options={"episode_index": idx}) if idx is not None else env_obj.reset()
    return out[0]

def _step(env_obj, action):
    obs, rew, term, trunc, info = env_obj.step(action)
    return obs, rew, bool(term or trunc), info

def collect_uc_trace(env_obj, policy_fn, instance_index=0):
    obs = _reset(env_obj, instance_index)
    done, trace = False, []
    while not done:
        a = policy_fn(obs)
        obs, rew, done, info = _step(env_obj, a)
        trace.append(info)
    return pd.DataFrame(trace)

def compact_state_for_gen(obs, gen_idx, n_gen, horizon=72, env_obj=None):
    e = env_obj if env_obj is not None else env
    lay = e.obs_layout

    if e.state_version == "v1":
        t = float(obs[0])
        prev_on = obs[slice(*lay["prev_on"])]
        nd_prof = obs[slice(*lay["net_demand"])]
        status  = obs[slice(*lay["status"])]
        look    = obs[slice(*lay["lookahead"])]
        t_idx   = min(int(t * (horizon - 1)), len(nd_prof) - 1)
        nd_now  = float(nd_prof[t_idx])
        fut     = nd_prof[t_idx:min(t_idx + 8, len(nd_prof))]
        fmin    = float(fut.min()) if len(fut) else nd_now
        return np.array([
            t, nd_now, float(nd_prof.max()), float(nd_prof.mean()),
            float(prev_on.sum()) / max(n_gen, 1),
            float(prev_on[gen_idx]), float(status[gen_idx]),
            fmin, float(look[3]),
            float(prev_on.sum()) / max(n_gen, 1),
            float(fmin - nd_now),
        ], dtype=float)

    sin_h, cos_h, remaining = obs[slice(*lay["time"])]
    win       = obs[slice(*lay["window"])]
    on_off    = obs[slice(*lay["on_off"])]
    can_start = obs[slice(*lay["can_start"])]
    can_stop  = obs[slice(*lay["can_stop"])]

    nd_now = float(win[0])
    fmin   = float(win.min())
    return np.array([
        float(sin_h), float(cos_h), float(remaining),
        nd_now, float(win.max()), float(win.mean()),
        float(on_off.sum()) / max(n_gen, 1),
        float(on_off[gen_idx]),
        float(can_start[gen_idx]),
        float(can_stop[gen_idx]),
        float(fmin - nd_now),
    ], dtype=float)


N_BINS = 6

def estimate_bounds(env_obj, n_samples=500):
    obs = _reset(env_obj)
    feats = []
    for _ in range(n_samples):
        a = np.random.randint(0, 2, env_obj.n_gen)
        obs, _, done, _ = _step(env_obj, a)
        feats.append(compact_state_for_gen(obs, 0, env_obj.n_gen, env_obj=env_obj))
        if done: obs = _reset(env_obj)
    arr = np.vstack(feats)
    lo  = np.quantile(arr, 0.02, axis=0)
    hi  = np.quantile(arr, 0.98, axis=0)
    return lo, np.where(hi <= lo + 1e-9, lo + 1.0, hi)


def discretize(x, lo, hi, n_bins=N_BINS):
    edges = np.linspace(lo, hi, n_bins + 1)[1:-1]
    return int(np.clip(np.digitize(x, edges), 0, n_bins - 1))


def discretize_state(obs, gen_idx):
    x = compact_state_for_gen(obs, gen_idx, env.n_gen, env_obj=env)
    return tuple(discretize(v, lo, hi) for v, lo, hi in zip(x, OBS_LO, OBS_HI))

In [ ]:
np.random.seed(42); random.seed(42)
env = UCOnlyEnv(train_episodes, spill_penalty=1000, reward_scale=1e5,
                random_reset=True, seed=42, lookahead=8,
                state_version="v2", obs_window=72)
print(f"Obs: {env.obs_dim} | Generators: {env.n_gen} | Horizon: {env.horizon}")

OBS_LO, OBS_HI = estimate_bounds(env)

## 3. Tabular Q-learning

In [ ]:
def build_edges(lo, hi, n_bins=N_BINS):
    return [np.linspace(l, h, n_bins + 1)[1:-1] for l, h in zip(lo, hi)]

EDGES = build_edges(OBS_LO, OBS_HI)


def discretize_all_gens(obs, env_obj=None):
    e = env_obj if env_obj is not None else env
    lay = e.obs_layout
    n = e.n_gen

    sin_h, cos_h, remaining = obs[slice(*lay["time"])]
    win       = obs[slice(*lay["window"])]
    on_off    = obs[slice(*lay["on_off"])]
    can_start = obs[slice(*lay["can_start"])]
    can_stop  = obs[slice(*lay["can_stop"])]

    nd_now   = float(win[0])
    win_max  = float(win.max())
    win_mean = float(win.mean())
    fmin     = float(win.min())
    frac_on  = float(on_off.sum()) / max(n, 1)

    g_vals = [float(sin_h), float(cos_h), float(remaining),
              nd_now, win_max, win_mean, frac_on]
    g_idx  = [0, 1, 2, 3, 4, 5, 6]
    g_bins = [int(np.clip(np.digitize(v, EDGES[k]), 0, N_BINS - 1))
              for v, k in zip(g_vals, g_idx)]

    delta = float(fmin - nd_now)
    d_bin = int(np.clip(np.digitize(delta, EDGES[10]), 0, N_BINS - 1))

    b_on    = np.clip(np.digitize(on_off,    EDGES[7]), 0, N_BINS - 1)
    b_start = np.clip(np.digitize(can_start, EDGES[8]), 0, N_BINS - 1)
    b_stop  = np.clip(np.digitize(can_stop,  EDGES[9]), 0, N_BINS - 1)

    g0, g1, g2, g3, g4, g5, g6 = g_bins
    return [(g0, g1, g2, g3, g4, g5, g6,
             int(b_on[i]), int(b_start[i]), int(b_stop[i]), d_bin)
            for i in range(n)]


def discretize_state(obs, gen_idx):
    return discretize_all_gens(obs)[gen_idx]


def q_policy_from_states(states, epsilon=0.0):
    n = len(states)
    action = np.empty(n, int)
    if epsilon > 0.0:
        explore = np.random.rand(n) < epsilon
        rand_a  = np.random.randint(0, 2, n)
    else:
        explore = np.zeros(n, bool); rand_a = None
    for i, s in enumerate(states):
        if explore[i]:
            action[i] = rand_a[i]
        else:
            q = Q_tables[i][s]
            action[i] = 0 if q[0] >= q[1] else 1
    return action


def q_policy(obs, epsilon=0.0):
    return q_policy_from_states(discretize_all_gens(obs), epsilon)

In [ ]:
Q_tables = [defaultdict(lambda: np.zeros(2)) for _ in range(env.n_gen)]

ALPHA, GAMMA = 0.10, 0.99                     
EPS, EPS_MIN, EPS_DECAY = 0.30, 0.05, 0.998
N_EPOCHS = 1 if SMOKE else 5
N_Q = min(5 if SMOKE else 300, len(train_episodes))
q_rewards, q_costs, q_startups = [], [], []

print(f"Training Tabular Q-Learning — {N_EPOCHS} epochs x {N_Q} episodes "
      f"= {N_EPOCHS*N_Q} total...")
t0 = time.time()
global_ep = 0
n_gen = env.n_gen

for epoch in range(N_EPOCHS):
    for ep in range(N_Q):
        obs = _reset(env)
        states = discretize_all_gens(obs)       
        done = False
        er = ec = esu = 0.0

        while not done:
            a = q_policy_from_states(states, EPS)
            nobs, rew, done, info = _step(env, a)

            if done:
                for i in range(n_gen):
                    q = Q_tables[i][states[i]]
                    ai = int(a[i])
                    q[ai] += ALPHA * (rew - q[ai])
                next_states = None
            else:
                next_states = discretize_all_gens(nobs)
                for i in range(n_gen):
                    q  = Q_tables[i][states[i]]
                    ai = int(a[i])
                    q[ai] += ALPHA * (rew + GAMMA * Q_tables[i][next_states[i]].max() - q[ai])

            states = next_states
            er += rew; ec += info["cost"]; esu += info["startup_cost"]

        q_rewards.append(er); q_costs.append(ec); q_startups.append(esu)
        EPS = max(EPS_MIN, EPS * EPS_DECAY)
        global_ep += 1
        if global_ep % 50 == 0:
            print(f"  [{time.time()-t0:5.0f}s] epoch {epoch+1}/{N_EPOCHS} "
                  f"ep {ep+1:03d} | rew={er:.2f} | startup=${esu:,.0f} | eps={EPS:.3f}")

print(f"Done in {time.time()-t0:.0f}s — {global_ep} total episodes")

## 4. Deep Q-network

In [ ]:
SEED = 42; np.random.seed(SEED); random.seed(SEED); torch.manual_seed(SEED)

GAMMA_DQN   = 0.99
BATCH_SIZE  = 64
TRAIN_EVERY = 2
WARMUP      = 1000
TGT_SYNC    = 500


class DQN(nn.Module):
    def __init__(self, state_dim, n_gen, hidden=256):
        super().__init__()
        self.n_gen = n_gen
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU())
        self.head = nn.Linear(hidden, n_gen * 2)

    def forward(self, x):
        h = self.shared(x)
        return self.head(h).view(-1, self.n_gen, 2)

    @torch.no_grad()
    def get_action(self, obs_np, epsilon=0.0):
        if np.random.rand() < epsilon:
            return np.random.randint(0, 2, self.n_gen)
        x = torch.as_tensor(obs_np, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        return self.forward(x).squeeze(0).argmax(dim=1).cpu().numpy()


STATE_DIM = env.obs_dim
pol_net = DQN(STATE_DIM, env.n_gen).to(DEVICE)
tgt_net = DQN(STATE_DIM, env.n_gen).to(DEVICE)
tgt_net.load_state_dict(pol_net.state_dict()); tgt_net.eval()
optimizer = torch.optim.Adam(pol_net.parameters(), lr=3e-4)


class ReplayBuf:
    def __init__(self, cap, state_dim, n_gen):
        self.cap = cap
        self.s  = np.zeros((cap, state_dim), np.float32)
        self.a  = np.zeros((cap, n_gen), np.int64)
        self.r  = np.zeros((cap, 1), np.float32)
        self.ns = np.zeros((cap, state_dim), np.float32)
        self.d  = np.zeros((cap, 1), np.float32)
        self.ptr = 0
        self.full = False

    def push(self, s, a, r, ns, d):
        i = self.ptr
        self.s[i]  = s
        self.a[i]  = a
        self.r[i]  = r
        self.ns[i] = ns
        self.d[i]  = d
        self.ptr = (i + 1) % self.cap
        if self.ptr == 0: self.full = True

    def __len__(self):
        return self.cap if self.full else self.ptr

    def sample(self, bs):
        idx = np.random.randint(0, len(self), bs)
        return (torch.as_tensor(self.s[idx],  device=DEVICE),
                torch.as_tensor(self.a[idx],  device=DEVICE),
                torch.as_tensor(self.r[idx],  device=DEVICE),
                torch.as_tensor(self.ns[idx], device=DEVICE),
                torch.as_tensor(self.d[idx],  device=DEVICE))

In [ ]:
rb = ReplayBuf(50000, STATE_DIM, env.n_gen)
gstep = 0; eps_d = 1.0
dqn_rewards, dqn_costs, dqn_startups, dqn_losses = [], [], [], []

N_EPOCHS_DQN = 1 if SMOKE else 5
N_EP_DQN = 20 if SMOKE else 300
ZERO_NS = np.zeros(STATE_DIM, np.float32)

print(f"Training DQN — {N_EPOCHS_DQN} epochs x 300 = {N_EPOCHS_DQN*N_EP_DQN} episodes...")
t0 = time.time()
global_ep = 0

for epoch in range(N_EPOCHS_DQN):
    for ep in range(N_EP_DQN):
        obs = _reset(env)
        done = False
        er = ec = esu = 0.0
        loss_sum = 0.0; loss_n = 0

        while not done:
            a = pol_net.get_action(obs, eps_d)
            nobs, rew, done, info = _step(env, a)
            ns = nobs if not done else ZERO_NS
            rb.push(obs, a, rew, ns, done)
            er += rew; ec += info["cost"]; esu += info["startup_cost"]
            obs = nobs; gstep += 1

            if len(rb) >= WARMUP and gstep % TRAIN_EVERY == 0:
                s_b, a_t, r_t, ns_t, d_t = rb.sample(BATCH_SIZE)
                q_sa = pol_net(s_b).gather(2, a_t.unsqueeze(2)).squeeze(2)
                with torch.no_grad():
                    target = r_t + GAMMA_DQN * (1 - d_t) * tgt_net(ns_t).max(2)[0]
                loss = F.mse_loss(q_sa, target)
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(pol_net.parameters(), 5.0)
                optimizer.step()
                loss_sum += loss.detach(); loss_n += 1

            if gstep % TGT_SYNC == 0:
                tgt_net.load_state_dict(pol_net.state_dict())

        dqn_rewards.append(er); dqn_costs.append(ec); dqn_startups.append(esu)
        dqn_losses.append(float(loss_sum / loss_n) if loss_n else np.nan)
        eps_d = max(0.05, eps_d * 0.998)
        global_ep += 1
        if global_ep % 50 == 0:
            print(f"  [{time.time()-t0:5.0f}s] epoch {epoch+1}/{N_EPOCHS_DQN} "
                  f"ep {ep+1:03d} | rew={er:.2f} | startup=${esu:,.0f} | eps={eps_d:.3f}")

print(f"Done in {time.time()-t0:.0f}s")


def dqn_policy(obs):
    return pol_net.get_action(obs, 0.0)

## 5. PPO baseline

In [ ]:
from stable_baselines3 import PPO as SB3_PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import BaseCallback

class StatsCB(BaseCallback):
    def __init__(self, tag="", every=100):
        super().__init__()
        self.tag, self.every = tag, every
        self.ep_r, self.ep_c, self.ep_su = [], [], []
        self._r, self._c, self._su = 0, 0, 0
        self.t0 = time.time()
    def _on_step(self):
        self._r  += float(self.locals["rewards"][0])
        self._c  += float(self.locals["infos"][0].get("cost", 0))
        self._su += float(self.locals["infos"][0].get("startup_cost", 0))
        if bool(self.locals["dones"][0]):
            self.ep_r.append(self._r); self.ep_c.append(self._c); self.ep_su.append(self._su)
            n = len(self.ep_r)
            if n % self.every == 0:
                print(f"  [{time.time()-self.t0:5.0f}s] {self.tag} ep {n:04d} | "
                      f"rew={self._r:.2f} | startup=${self._su:,.0f}")
            self._r, self._c, self._su = 0, 0, 0
        return True

class UCGymWrapper(gym.Env):
    metadata = {"render_modes": []}
    def __init__(self, episodes, seed=42, state_version="v2", obs_window=72, spill_penalty=1000):
        super().__init__()
        self.base = UCOnlyEnv(episodes, spill_penalty=spill_penalty, reward_scale=1e5,
                              random_reset=True, seed=seed, lookahead=8,
                              state_version=state_version, obs_window=obs_window)
        self.observation_space = spaces.Box(-np.inf, np.inf,
                                            shape=(self.base.obs_dim,), dtype=np.float32)
        self.action_space = spaces.MultiBinary(self.base.n_gen)
    def reset(self, *, seed=None, options=None):
        if seed: super().reset(seed=seed)
        return self.base.reset(options=options)
    def step(self, action): return self.base.step(action)


def train_ppo(episodes, state_version="v2", obs_window=72, gamma=0.99,
              n_epochs_data=5, seed=42, verbose_tag=""):
    cb = StatsCB()
    venv = DummyVecEnv([lambda: Monitor(
        UCGymWrapper(episodes, seed=seed,
                     state_version=state_version, obs_window=obs_window))])
    total_ts = max(512, int(300 * 72 * n_epochs_data * PPO_SCALE))
    model = SB3_PPO("MlpPolicy", venv, verbose=0,
                    n_steps=256, batch_size=64, gamma=gamma,
                    gae_lambda=0.95, learning_rate=3e-4, ent_coef=0.01,
                    clip_range=0.2, n_epochs=10, seed=seed, device=str(DEVICE))
    print(f"Training PPO [{verbose_tag}] — state={state_version} | W={obs_window} | "
          f"gamma={gamma} | {total_ts:,} timesteps (~{total_ts//72} episodes)...")
    t0 = time.time()
    model.learn(total_timesteps=total_ts, callback=cb)
    print(f"Done in {time.time()-t0:.0f}s | {len(cb.ep_r)} episodes")
    return model, cb

In [ ]:
ppo_v2, cb_v2 = train_ppo(train_episodes, "v2", 72, 0.99,
                          n_epochs_data=5, seed=42, verbose_tag="v2-1000")
ppo_v2.save(OUT / "saved_models_ppo" / "ppo_a_v2_sp1000")

## 6. Save checkpoints

In [ ]:
torch.save({"state_dict": pol_net.state_dict(),
            "state_dim": STATE_DIM, "n_gen": env.n_gen},
           OUT / "saved_models_dqn" / "dqn.pt")
for f in sorted(OUT.rglob("*")):
    if f.is_file():
        print(f"  {str(f.relative_to(OUT)):45s} {f.stat().st_size/1024:.0f} KB")

## 7. Training curves

In [ ]:
def plot_training_curves(rewards, costs, startups, title, losses=None, window=10):
    n = 4 if losses is not None else 3
    fig, axes = plt.subplots(1, n, figsize=(6*n, 5))
    fig.suptitle(title, fontsize=14, y=1.02)
    for ax, (data, lbl, clr) in zip(axes, [
        (rewards,"Episode Reward","tab:blue"),
        (costs,"Cost Proxy ($)","tab:red"),
        (startups,"Startup Cost ($)","tab:purple"),
    ]):
        ax.plot(data, alpha=0.2, color=clr)
        ax.plot(pd.Series(data).rolling(window, min_periods=1).mean(), color=clr, lw=2)
        ax.set_title(lbl); ax.set_xlabel("episode"); ax.grid(True, alpha=0.3)
    if losses is not None:
        ax = axes[3]
        ax.plot(losses, alpha=0.2, color="tab:green")
        ax.plot(pd.Series(losses).rolling(window, min_periods=1).mean(), color="tab:green", lw=2)
        ax.set_title("DQN Loss"); ax.set_xlabel("episode"); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

plot_training_curves(q_rewards, q_costs, q_startups, "Q-Learning UC Training")
plot_training_curves(dqn_rewards, dqn_costs, dqn_startups, "DQN UC Training", dqn_losses)
plot_training_curves(cb_v2.ep_r, cb_v2.ep_c, cb_v2.ep_su, "PPO UC Training")

## 8. Forecast-window ablation

In [ ]:
ABLATION_WINDOWS = [1, 4, 8, 24, 72]

def run_window_ablation(train_eps, test_eps, windows=ABLATION_WINDOWS,
                        n_epochs_data=3, seed=42, save_dir=OUT / "ablation"):
    from pathlib import Path
    out = Path(save_dir); out.mkdir(exist_ok=True)
    results, curves = [], {}
    for W in windows:
        model, cb = train_ppo(train_eps, "v2", W, 0.99,
                              n_epochs_data=n_epochs_data, seed=seed,
                              verbose_tag=f"W={W}")
        model.save(out / f"ppo_W{W}")
        curves[W] = {"rew": cb.ep_r, "cost": cb.ep_c, "startup": cb.ep_su}

        eval_env = UCOnlyEnv(test_eps, random_reset=False,
                             state_version="v2", obs_window=W)
        pol = lambda o: np.asarray(model.predict(o, deterministic=True)[0]).ravel()
        costs, unserved, online, first12 = [], [], [], []
        for i in range(len(test_eps)):
            tr = collect_uc_trace(eval_env, pol, instance_index=i)
            costs.append(tr["cost"].sum())
            unserved.append((tr["unserved_proxy"] > 1e-6).mean())
            online.append(tr["n_online"].mean())
            first12.append(tr["n_online"].values[:12].mean())
        results.append({
            "window": W,
            "mean_cost": float(np.mean(costs)),
            "unserved_rate": float(np.mean(unserved)),
            "mean_online": float(np.mean(online)),
            "mean_online_first12h": float(np.mean(first12)),
        })
        print(f"  W={W}: cost={results[-1]['mean_cost']:,.0f} "
              f"online={results[-1]['mean_online']:.1f} "
              f"first12h={results[-1]['mean_online_first12h']:.1f}")

    df = pd.DataFrame(results)
    df.to_csv(out / "window_ablation.csv", index=False)
    with open(out / "curves.pkl", "wb") as f:
        pickle.dump(curves, f)
    return df, curves

In [ ]:
WINDOWS = ABLATION_WINDOWS[:2] if SMOKE else ABLATION_WINDOWS
MODELDIR = OUT / "ablation"


def window_breakdown(test_eps, windows=WINDOWS, spill_penalty=1000):
    rows = []
    for W in windows:
        model = SB3_PPO.load(MODELDIR / f"ppo_W{W}")
        e = UCOnlyEnv(test_eps, spill_penalty=spill_penalty, random_reset=False,
                      state_version="v2", obs_window=W)
        pol = lambda o: np.asarray(model.predict(o, deterministic=True)[0]).ravel()

        corr, amp, marg, uns, spl, su, cost, onl = [], [], [], [], [], [], [], []
        for i in range(len(test_eps)):
            tr = collect_uc_trace(e, pol, instance_index=i)
            n_on = tr["n_online"].to_numpy(float)
            nd   = tr["net_demand"].to_numpy(float)

            if n_on.std() > 1e-9 and nd.std() > 1e-9:
                corr.append(np.corrcoef(n_on, nd)[0, 1])
            amp.append(n_on.std())

            req = np.maximum(nd, 0.0)
            oc  = tr["over_commit"].to_numpy(float)
            marg.append(np.mean(oc / np.maximum(req, 1.0)))

            uns.append(tr["unserved_proxy"].sum())
            spl.append(oc.sum())
            su.append(tr["startup_cost"].sum())
            cost.append(tr["cost"].sum())
            onl.append(n_on.mean())

        rows.append({
            "W": W,
            "responsiveness": np.mean(corr) if corr else np.nan,
            "amplitude_units": np.mean(amp),
            "reserve_margin": np.mean(marg),
            "mean_online": np.mean(onl),
            "unserved_MWh": np.mean(uns),
            "spill_MWh": np.mean(spl),
            "startup_cost_$": np.mean(su),
            "total_cost_$": np.mean(cost),
        })
        print(f"  W={W:2d} done — corr={rows[-1]['responsiveness']:.3f} "
              f"amp={rows[-1]['amplitude_units']:.2f}")

    df = pd.DataFrame(rows)
    df.to_csv(MODELDIR / "window_breakdown.csv", index=False)
    return df

def show_breakdown(df):
    out = pd.DataFrame({
        "W": df["W"],
        "Responsiveness (corr)": df["responsiveness"].round(3),
        "Amplitude (units, sd)": df["amplitude_units"].round(2),
        "Reserve margin": (df["reserve_margin"] * 100).round(1).astype(str) + "%",
        "Mean units online": df["mean_online"].round(1),
        "Unserved (MWh/ep)": df["unserved_MWh"].round(1),
        "Spill (MWh/ep)": df["spill_MWh"].round(0).astype(int),
        "Startup ($/ep)": (df["startup_cost_$"] / 1000).round(0).astype(int).astype(str) + "k",
        "Total cost ($M/ep)": (df["total_cost_$"] / 1e6).round(2),
    })
    print(out.to_string(index=False))
    return out

In [ ]:
df_abl, curves_abl = run_window_ablation(train_episodes, test_episodes[:N_EVAL], windows=WINDOWS)

In [ ]:
df_bd = window_breakdown(test_episodes[:N_EVAL], windows=WINDOWS)
tbl = show_breakdown(df_bd)

## 9. Observation design: v1 against v2

In [ ]:
def run_episode_detailed(env_obj, policy, instance_index=0):
    obs, _ = env_obj.reset(options={"episode_index": instance_index})
    on, raw, rows = [], [], []
    done = False
    while not done:
        a = policy(obs)
        obs, rew, done, _, info = env_obj.step(a)
        on.append(info["on"]); raw.append(info["raw_action"])
        rows.append({"hour": info["hour"], "n_online": info["n_online"],
                     "n_overrides": info["n_overrides"],
                     "over_commit": info["over_commit"],
                     "unserved": info["unserved_proxy"],
                     "net_demand": info["net_demand"],
                     "n_startups": info["n_startups"], "cost": info["cost"]})
    on = np.array(on); raw = np.array(raw)
    return {"on": on, "raw": raw, "illegal": (on != raw),
            "df": pd.DataFrame(rows)}

def compare_legality(models_by_label, test_eps, n_instances=20,
                     spill_penalty=1000):
    rows = []
    for label, (model, sv, W) in models_by_label.items():
        e = UCOnlyEnv(test_eps, spill_penalty=spill_penalty, random_reset=False,
                      state_version=sv, obs_window=W)
        pol = lambda o: np.asarray(model.predict(o, deterministic=True)[0]).ravel()
        ov, ill, cost, online = [], [], [], []
        for i in range(min(n_instances, len(test_eps))):
            tr = run_episode_detailed(e, pol, instance_index=i)
            ov.append(tr["df"]["n_overrides"].sum())
            ill.append(tr["illegal"].mean())
            cost.append(tr["df"]["cost"].sum())
            online.append(tr["df"]["n_online"].mean())
        rows.append({"state": label,
                     "overrides_per_episode": np.mean(ov),
                     "illegal_rate": np.mean(ill),
                     "mean_cost": np.mean(cost),
                     "mean_online": np.mean(online)})
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    return df

In [ ]:
ppo_v1, cb_v1 = train_ppo(train_episodes, "v1", 72, 0.95,
                          n_epochs_data=5, seed=42, verbose_tag="v1")
df_legal = compare_legality(
    {"v1 (prev_on only)":      (ppo_v1, "v1", 72),
     "v2 (constraint clocks)": (ppo_v2, "v2", 72)},
    test_episodes, n_instances=N_EVAL or 20, spill_penalty=1000)
df_legal.to_csv(OUT / "v1_vs_v2.csv", index=False)

## 10. State representation and spill penalty (2 x 2)

In [ ]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces


class UCOnlyEnvV3(UCOnlyEnv):

    def __init__(self, *args, **kwargs):
        kwargs["state_version"] = "v2"
        self._v3_ready = False
        self.starts_so_far = None
        self.prev_spill = 0.0
        self.prev_unserved = 0.0
        super().__init__(*args, **kwargs)

        self._max_starts = np.maximum(
            self.horizon / np.maximum(self.min_up + self.min_down, 1.0), 1.0)
        self._v3_ready = True
        self.state_version = "v3"
        self.obs_dim = 3 + self.obs_window + 4 + 3 * self.n_gen + 4 + self.n_gen
        self.reset(options={"episode_index": 0})

    @property
    def obs_layout(self):
        n, W, o = self.n_gen, self.obs_window, 0
        lay = {}
        for key, size in [("time", 3), ("window", W), ("lookahead", 4),
                          ("on_off", n), ("can_start", n), ("can_stop", n),
                          ("balance", 4), ("starts", n)]:
            lay[key] = (o, o + size); o += size
        return lay

    def reset(self, seed=None, options=None):
        self.starts_so_far = np.zeros(self.n_gen, float)
        self.prev_spill = 0.0
        self.prev_unserved = 0.0
        self.state_version = "v2"
        obs, info = super().reset(seed=seed, options=options)
        if not self._v3_ready:
            return obs, info
        self.state_version = "v3"
        return self._obs(), info

    def _balance_features(self):
        denom = max(float(self.net_demand[self.t]), 1.0)
        on_m = (self.current_dispatch > 1e-6) | (self.status_duration > 0)
        return np.array([
            np.clip(float(self.pmin[on_m].sum()) / denom, 0.0, 3.0),
            np.clip(float(self.pmax[on_m].sum()) / denom, 0.0, 3.0),
            np.clip(self.prev_spill / denom, 0.0, 3.0),
            np.clip(self.prev_unserved / denom, 0.0, 3.0),
        ], dtype=np.float32)

    def _obs(self):
        if not self._v3_ready:
            return super()._obs()
        self.state_version = "v2"
        base = super()._obs()
        self.state_version = "v3"
        return np.concatenate([
            base,
            self._balance_features(),
            np.clip(self.starts_so_far / self._max_starts, 0.0, 1.0),
        ]).astype(np.float32)

    def step(self, action):
        cur_on = (self.current_dispatch > 1e-6) | (self.status_duration > 0)
        self.state_version = "v2"
        obs, rew, done, trunc, info = super().step(action)
        self.state_version = "v3"
        self.starts_so_far = self.starts_so_far + \
            ((info["on"] == 1) & (~cur_on)).astype(float)
        self.prev_spill = info["over_commit"]
        self.prev_unserved = info["unserved_proxy"]
        obs = np.zeros(self.obs_dim, np.float32) if done else self._obs()
        return obs, rew, done, trunc, info


class UCGymWrapperV3(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self, episodes, seed=42, obs_window=72, spill_penalty=1000):
        super().__init__()
        self.base = UCOnlyEnvV3(episodes, spill_penalty=spill_penalty,
                                reward_scale=1e5, random_reset=True, seed=seed,
                                lookahead=8, obs_window=obs_window)
        self.observation_space = spaces.Box(
            -np.inf, np.inf, shape=(self.base.obs_dim,), dtype=np.float32)
        self.action_space = spaces.MultiBinary(self.base.n_gen)

    def reset(self, *, seed=None, options=None):
        if seed: super().reset(seed=seed)
        return self.base.reset(options=options)

    def step(self, action):
        return self.base.step(action)


_e = UCOnlyEnvV3(train_episodes, spill_penalty=1000, obs_window=72)
_o, _ = _e.reset(options={"episode_index": 0})
assert _o.shape[0] == _e.obs_dim, (_o.shape, _e.obs_dim)
for _ in range(5):
    _o, _r, _d, _, _i = _e.step(np.random.randint(0, 2, _e.n_gen))
print("v3 OK | obs_dim:", _e.obs_dim, "| all finite:", bool(np.all(np.isfinite(_o))))
print("balance feats [pmin_ratio, pmax_ratio, spill, unserved]:",
      np.round(_o[slice(*_e.obs_layout["balance"])], 3))
print("starts range:", _o[slice(*_e.obs_layout["starts"])].min(),
      "-", _o[slice(*_e.obs_layout["starts"])].max())

In [ ]:
SAVEDIR = OUT / "saved_models_ppo"
TOTAL_TS = max(512, int(300 * 72 * 5 * PPO_SCALE))
PPO_KW = dict(n_steps=256, batch_size=64, gamma=0.99, gae_lambda=0.95,
              learning_rate=3e-4, ent_coef=0.01, clip_range=0.2,
              n_epochs=10, seed=42, verbose=0, device=str(DEVICE))


def _train(make_env, tag):
    cb = StatsCB(tag=tag) if "tag" in StatsCB.__init__.__code__.co_varnames \
         else StatsCB()
    venv = DummyVecEnv([lambda: Monitor(make_env())])
    model = SB3_PPO("MlpPolicy", venv, **PPO_KW)
    print(f"Training PPO [{tag}] — {TOTAL_TS:,} steps "
          f"(obs_dim={venv.observation_space.shape[0]})...")
    t0 = time.time()
    model.learn(total_timesteps=TOTAL_TS, callback=cb)
    print(f"  done in {time.time()-t0:.0f}s | {len(cb.ep_r)} episodes")
    model.save(SAVEDIR / f"ppo_{tag}")
    return model, cb

def evaluate(model, env_obj, n_inst=None):
    pol = lambda o: np.asarray(model.predict(o, deterministic=True)[0]).ravel()
    n = n_inst or (N_EVAL or len(test_episodes))
    corr, amp, uns, spl, su, cost, onl, ovr = [], [], [], [], [], [], [], []
    for i in range(n):
        tr = collect_uc_trace(env_obj, pol, instance_index=i)
        n_on = tr["n_online"].to_numpy(float)
        nd = tr["net_demand"].to_numpy(float)
        if n_on.std() > 1e-9 and nd.std() > 1e-9:
            corr.append(np.corrcoef(n_on, nd)[0, 1])
        amp.append(n_on.std())
        uns.append(tr["unserved_proxy"].sum())
        spl.append(tr["over_commit"].sum())
        su.append(tr["startup_cost"].sum())
        cost.append(tr["cost"].sum())
        onl.append(n_on.mean())
        ovr.append(tr["n_overrides"].sum())
    return {"responsiveness": np.mean(corr), "amplitude": np.mean(amp),
            "mean_online": np.mean(onl), "unserved_MWh": np.mean(uns),
            "spill_MWh": np.mean(spl), "startup_$": np.mean(su),
            "cost_$M": np.mean(cost) / 1e6, "overrides": np.mean(ovr)}

In [ ]:
ppo_v3, cb_v3 = _train(
    lambda: UCGymWrapperV3(train_episodes, seed=42, obs_window=72,
                           spill_penalty=1000), "c_v3_sp1000")
ppo_v4, cb_v4 = _train(
    lambda: UCGymWrapperV3(train_episodes, seed=42, obs_window=72,
                           spill_penalty=3000), "d_v3_sp3000")
ppo_v2b, cb_v2b = _train(
    lambda: UCGymWrapper(train_episodes, seed=42, state_version="v2",
                         obs_window=72, spill_penalty=3000), "b_v2_sp3000")

In [ ]:
EVAL_SP = 1000

res = pd.DataFrame([
    {"state": "v2, trained sp=1000", **evaluate(ppo_v2, UCOnlyEnv(
        test_episodes, spill_penalty=EVAL_SP, random_reset=False,
        state_version="v2", obs_window=72))},
    {"state": "v3, trained sp=1000", **evaluate(ppo_v3, UCOnlyEnvV3(
        test_episodes, spill_penalty=EVAL_SP, random_reset=False,
        obs_window=72))},
    {"state": "v3, trained sp=3000", **evaluate(ppo_v4, UCOnlyEnvV3(
        test_episodes, spill_penalty=EVAL_SP, random_reset=False,
        obs_window=72))},
    {"state": "v2, trained sp=3000", **evaluate(ppo_v2b, UCOnlyEnv(
        test_episodes, spill_penalty=EVAL_SP, random_reset=False,
        state_version="v2", obs_window=72))},
]).set_index("state").round(3)
print(res.to_string())
res.to_csv(OUT / "factorial_single_seed.csv")